In [46]:
from src.data.data_prep import scrape_all_years, scrape_tourney_results
import os
import pandas as pd
import numpy as np


In [47]:
print("=== Scraping Barttorvik ===")
#stores data in data/processed/barttorvik_team_stats.csv
scrape_all_years()
#assert that barttorvik team stats file exists
assert os.path.isfile("data/processed/barttorvik_team_stats.csv"), "File data/processed/barttorvik_team_stats.csv not found"
#stores data
print("=== Scraping Sports Reference Results ===")
scrape_tourney_results()
#assert that results file exists
assert os.path.isfile("data/processed/sports_ref_team_results.csv"), "File data/processed/sports_ref_team_results.csv not found"

=== Scraping Barttorvik ===
Fetching 2010...
Fetching 2011...
Fetching 2012...
Fetching 2013...
Fetching 2014...
Fetching 2015...
Fetching 2016...
Fetching 2017...
Fetching 2018...
Fetching 2019...
Fetching 2020...
Fetching 2021...
Fetching 2022...
Fetching 2023...
Fetching 2024...
Fetching 2025...
=== Scraping Sports Reference Results ===
Fetching 2010...
Fetching 2011...
Fetching 2012...
Fetching 2013...
Fetching 2014...
Fetching 2015...
Fetching 2016...
Fetching 2017...
Fetching 2018...
Fetching 2019...
Fetching 2020...
Fetching 2021...
Fetching 2022...
Fetching 2023...
Fetching 2024...
Fetching 2025...


In [53]:
team_stats_df = pd.read_csv("data/processed/barttorvik_team_stats.csv")
results_df = pd.read_csv("data/processed/sports_ref_team_results.csv")
#accounting for covid season (2020 is dropped because no tournament but there aree stats for it)
seasons = pd.concat([results_df["season"], team_stats_df["season"]]).drop_duplicates()
print(type(seasons))
team_stats_df = team_stats_df[team_stats_df["season"].isin(seasons)]
results_df = results_df[results_df["season"].isin(seasons)]
# Build spelling → TeamID lookup (all keys lowercased)
spellings = pd.read_csv("MTeamSpellings.csv")
name_to_id = dict(zip(spellings["TeamNameSpelling"].str.lower(), spellings["TeamID"]))
for season in seasons:
    mm_teams = results_df[results_df["season"] == int(season)][["team_winner", "team_loser"]]
    #stack winner and loser columns on top of each other, then get unique values
    mm_teams = list(mm_teams.stack().drop_duplicates())
    teams_from_stats = list(team_stats_df[team_stats_df["season"] == int(season)]["team"].unique())
    #for all teams from results
    for team in teams_from_stats:
        #if lowercase team name is not a key in name_to_id, print the team name and season
        if team.lower() not in name_to_id:
            print(f"no match for {team} in season {int(season)}")
#Need to convert all team names to lowercase because that's what name_to_id uses as keys         
team_stats_df["team_id"] = team_stats_df["team"].str.lower().map(name_to_id)
results_df["team_winner_id"] = results_df["team_winner"].str.lower().map(name_to_id)
results_df["team_loser_id"] = results_df["team_loser"].str.lower().map(name_to_id)

<class 'pandas.Series'>


In [56]:
# All numeric stat columns from barttorvik (unk_* kept since values are non-zero)
STAT_COLS = [
    "adjoe", "adjde", "barthag", "wins", "games_played",
    "efg_o", "efg_d", "ftr", "ftrd", "tor", "tord",
    "orb", "drb", "adj_t", "twop_o", "twop_d",
    "threep_o", "threep_d", "unk_3", "unk_4",
    "threer_o", "threer_d", "wab", "unk_11", "adj_em"
]

#making two dataframes so we can merge onto results_df to form matchups dataframe
winner_stats = (
    team_stats_df[["team_id", "season"] + STAT_COLS]
    .rename(columns={c: f"w_{c}" for c in STAT_COLS})
)
loser_stats = (
    team_stats_df[["team_id", "season"] + STAT_COLS]
    .rename(columns={c: f"l_{c}" for c in STAT_COLS})
)


In [57]:
print(team_stats_df["team_id"].head())
print(winner_stats["team_id"].head())

0    1364
1    1115
2    1405
3    1231
4    1418
Name: team_id, dtype: int64
0    1364
1    1115
2    1405
3    1231
4    1418
Name: team_id, dtype: int64


In [58]:
print(results_df.shape)
print(winner_stats.shape)
print(loser_stats.shape)
print(results_df.memory_usage(deep=True).sum() / 1e6, "MB")
print(winner_stats.duplicated(subset=["team_id", "season"]).sum())
print(loser_stats.duplicated(subset=["team_id", "season"]).sum())

(914, 10)
(5639, 27)
(5639, 27)
0.206052 MB
0
0


In [59]:

#dropping team id because we don't need it because we already have the winner and loser ids
matchups = (
    results_df
    .merge(winner_stats, left_on=["team_winner_id", "season"], right_on=["team_id", "season"], how="inner")
    .drop(columns=["team_id"])
    .merge(loser_stats, left_on=["team_loser_id", "season"], right_on=["team_id", "season"], how="inner")
    .drop(columns=["team_id"])
)

print(f"Games after stat merge: {len(matchups)} (dropped {len(results_df) - len(matchups)} with missing stats)")

# consistent team ordering: team_1 = lower seed number (better seeded)
# at inference time we don't know the winner, so we order by seed instead such that t1 is always first
# margin = t1_score - t2_score  -> negative means upset
is_t1_winner = matchups["seed_winner"] <= matchups["seed_loser"]

#if t1 (lower seed) is the winner, then t1_seed is the seed of the winner, otherwise it is the seed of the loser
#same thing for t2 and for two score columns
#TODO maybe simplify this? Seems like there should be an easier way to do this than have four different lines of code
matchups["t1_seed"]  = np.where(is_t1_winner, matchups["seed_winner"],  matchups["seed_loser"])
matchups["t2_seed"]  = np.where(is_t1_winner, matchups["seed_loser"],   matchups["seed_winner"])
matchups["t1_score"] = np.where(is_t1_winner, matchups["score_winner"], matchups["score_loser"])
matchups["t2_score"] = np.where(is_t1_winner, matchups["score_loser"],  matchups["score_winner"])

for col in STAT_COLS:
    matchups[f"t1_{col}"] = np.where(is_t1_winner, matchups[f"w_{col}"], matchups[f"l_{col}"])
    matchups[f"t2_{col}"] = np.where(is_t1_winner, matchups[f"l_{col}"], matchups[f"w_{col}"])

# ---LABELS ---
#total points is the sum of the two scores
matchups["total_points"]   = matchups["t1_score"] + matchups["t2_score"]
#Therefore winning margin is (lower seed score) - (higher seed score)
matchups["winning_margin"] = matchups["t1_score"] - matchups["t2_score"]

# seed_diff: positive means t1 is favored (e.g. seed 1 vs seed 16 → diff = 15)
matchups["seed_diff"] = matchups["t2_seed"] - matchups["t1_seed"]


Games after stat merge: 914 (dropped 0 with missing stats)


In [63]:

# ── Feature matrix and label vectors ──
FEATURE_COLS = (
    ["t1_seed", "t2_seed"]
    + [f"t1_{c}" for c in STAT_COLS]
    + [f"t2_{c}" for c in STAT_COLS]
)
LABEL_COLS = ["total_points", "winning_margin"]

# ── Train / val / test split by season (no shuffle — prevents leakage) ──
# 2020 had no tournament (COVID), so effective seasons: 2010-2019, 2021-2025
train_seasons = list(range(2010, 2023))   # 10 tournament years
val_seasons   = [2023, 2024]
test_seasons  = [2025]

train_df = matchups[matchups["season"].isin(train_seasons)]
val_df   = matchups[matchups["season"].isin(val_seasons)]
test_df  = matchups[matchups["season"].isin(test_seasons)]

X_train, y_train = train_df[FEATURE_COLS].values, train_df[LABEL_COLS].values
X_val,   y_val   = val_df[FEATURE_COLS].values,   val_df[LABEL_COLS].values
X_test,  y_test  = test_df[FEATURE_COLS].values,  test_df[LABEL_COLS].values

print(f"Features: {len(FEATURE_COLS)}")
print(f"Train: {X_train.shape}  ({sorted(train_df['season'].unique())})")
print(f"Val:   {X_val.shape}    ({sorted(val_df['season'].unique())})")
print(f"Test:  {X_test.shape}   ({sorted(test_df['season'].unique())})")


Features: 52
Train: (725, 52)  ([np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2021), np.int64(2022)])
Val:   (126, 52)    ([np.int64(2023), np.int64(2024)])
Test:  (63, 52)   ([np.int64(2025)])


In [66]:
# Dumb baselines on the same test split (compare to your GBM metrics)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

y_train_arr = train_df[LABEL_COLS].values
y_test_arr = test_df[LABEL_COLS].values

print("=== Baseline 1: constant = train-set mean (predict same value every test game) ===")
for j, name in enumerate(LABEL_COLS):
    mu = float(y_train_arr[:, j].mean())
    pred = np.full(len(test_df), mu)
    rmse = np.sqrt(mean_squared_error(y_test_arr[:, j], pred))
    mae = mean_absolute_error(y_test_arr[:, j], pred)
    print(f"{name}: mean={mu:.3f}  |  RMSE={rmse:.3f}  MAE={mae:.3f}")

print("\n=== Baseline 2: OLS on seed_diff only (fit train, predict test) ===")
X_tr_sd = train_df[["seed_diff"]].values
X_te_sd = test_df[["seed_diff"]].values
for j, name in enumerate(LABEL_COLS):
    lr = LinearRegression()
    lr.fit(X_tr_sd, y_train_arr[:, j])
    pred = lr.predict(X_te_sd)
    rmse = np.sqrt(mean_squared_error(y_test_arr[:, j], pred))
    mae = mean_absolute_error(y_test_arr[:, j], pred)
    print(
        f"{name}: y ≈ {lr.intercept_:.3f} + {lr.coef_[0]:.4f} * seed_diff  |  "
        f"RMSE={rmse:.3f}  MAE={mae:.3f}"
    )

=== Baseline 1: constant = train-set mean (predict same value every test game) ===
total_points: mean=138.492  |  RMSE=19.043  MAE=14.455
winning_margin: mean=6.594  |  RMSE=13.163  MAE=10.258

=== Baseline 2: OLS on seed_diff only (fit train, predict test) ===
total_points: y ≈ 137.722 + 0.1189 * seed_diff  |  RMSE=19.111  MAE=14.475
winning_margin: y ≈ -1.189 + 1.2014 * seed_diff  |  RMSE=11.587  MAE=9.295


In [64]:
# LightGBM hyperparameter search (train → select on val; time-ordered splits unchanged)
import lightgbm as lgb
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Shared fixed settings (objective, high n_estimators; early stopping picks effective round count)
LGBM_FIXED = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "verbosity": -1,
    "random_state": 42,
    "n_estimators": 2000,
    "bagging_freq": 5,
}

# Random search space (narrow freely once you see what works)
param_grid = {
    "num_leaves": [15, 31, 47, 63],
    "max_depth": [-1, 6, 12],
    "learning_rate": [0.02, 0.05, 0.08, 0.12],
    "min_child_samples": [5, 15, 25, 40],
    "feature_fraction": [0.7, 0.85, 1.0],
    "bagging_fraction": [0.7, 0.85, 1.0],
    "reg_alpha": [0.0, 0.05, 0.1],
    "reg_lambda": [0.0, 0.05, 0.1, 0.2],
}

N_ITER = 40
STEPS = 80  # early stopping patience

# sklearn ParameterSampler only accepts int or numpy.RandomState, not np.random.Generator
SEARCH_SEED = 42

# Separate search per target: minimize val RMSE for total_points, then for winning_margin
best_models = []
best_params_per_target = {}

#Looping to find two distinct models for total score AND margin
for j, name in enumerate(LABEL_COLS):
    best_val_rmse = np.inf
    best_params_j = None
    best_model_j = None

    for sampled in ParameterSampler(param_grid, n_iter=N_ITER, random_state=SEARCH_SEED + j):
        trial = {**LGBM_FIXED, **sampled}
        m = lgb.LGBMRegressor(**trial)
        m.fit(
            X_train,
            y_train[:, j],
            eval_set=[(X_val, y_val[:, j])],
            callbacks=[lgb.early_stopping(STEPS, verbose=False)],
        )
        pred_v = m.predict(X_val)
        val_rmse = float(np.sqrt(mean_squared_error(y_val[:, j], pred_v)))
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            best_params_j = {
                **trial,
                "best_iteration": getattr(m, "best_iteration_", None),
            }
            best_model_j = m

    best_models.append(best_model_j)
    best_params_per_target[name] = best_params_j

    print(f"\n{name} — best val RMSE: {best_val_rmse:.4f}")
    print("  params:")
    for k in sorted(param_grid.keys()):
        print(f"    {k}: {best_params_j[k]}")

# Test-set metrics (each target uses its own best model)
for j, name in enumerate(LABEL_COLS):
    pred_te = best_models[j].predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test[:, j], pred_te))
    mae = mean_absolute_error(y_test[:, j], pred_te)
    print(f"{name}: test RMSE={rmse:.3f}  MAE={mae:.3f}")


total_points — best val RMSE: 16.5472
  params:
    bagging_fraction: 0.85
    feature_fraction: 0.85
    learning_rate: 0.12
    max_depth: 12
    min_child_samples: 15
    num_leaves: 31
    reg_alpha: 0.05
    reg_lambda: 0.1

winning_margin — best val RMSE: 9.5496
  params:
    bagging_fraction: 0.7
    feature_fraction: 0.85
    learning_rate: 0.08
    max_depth: 12
    min_child_samples: 40
    num_leaves: 63
    reg_alpha: 0.0
    reg_lambda: 0.0
total_points: test RMSE=15.297  MAE=11.226
winning_margin: test RMSE=9.805  MAE=8.197


In [65]:
# XGBoost hyperparameter search (same protocol as LightGBM: per-target val RMSE, then test)
import xgboost as xgb
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# XGBoost analogs of the LightGBM grid (num_leaves → depth / min_child_weight)
param_grid_xgb = {
    "max_depth": [4, 6, 10, 12],
    "learning_rate": [0.02, 0.05, 0.08, 0.12],
    "min_child_weight": [1, 5, 10, 20],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "reg_alpha": [0.0, 0.05, 0.1],
    "reg_lambda": [0.0, 0.05, 0.1, 0.2],
}

N_ITER_XGB = 40
STEPS_XGB = 80
SEARCH_SEED_XGB = 43  # different seed from LightGBM search for independent samples

XGB_FIXED = {
    "objective": "reg:squarederror",
    "tree_method": "hist",
    "n_estimators": 2000,
    "random_state": 42,
    "verbosity": 0,
    "early_stopping_rounds": STEPS_XGB,
}

best_models_xgb = []
best_params_xgb = {}

for j, name in enumerate(LABEL_COLS):
    best_val_rmse = np.inf
    best_params_j = None
    best_model_j = None

    for sampled in ParameterSampler(param_grid_xgb, n_iter=N_ITER_XGB, random_state=SEARCH_SEED_XGB + j):
        trial = {**XGB_FIXED, **sampled}
        m = xgb.XGBRegressor(**trial)
        m.fit(
            X_train,
            y_train[:, j],
            eval_set=[(X_val, y_val[:, j])],
            verbose=False,
        )
        pred_v = m.predict(X_val)
        val_rmse = float(np.sqrt(mean_squared_error(y_val[:, j], pred_v)))
        if val_rmse < best_val_rmse:
            best_val_rmse = val_rmse
            bi = getattr(m, "best_iteration", None)
            if bi is None:
                bi = getattr(m, "best_iteration_", None)
            best_params_j = {**trial, "best_iteration": bi}
            best_model_j = m

    best_models_xgb.append(best_model_j)
    best_params_xgb[name] = best_params_j

    print(f"\n{name} — best val RMSE: {best_val_rmse:.4f}")
    print("  params:")
    for k in sorted(param_grid_xgb.keys()):
        print(f"    {k}: {best_params_j[k]}")

for j, name in enumerate(LABEL_COLS):
    pred_te = best_models_xgb[j].predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test[:, j], pred_te))
    mae = mean_absolute_error(y_test[:, j], pred_te)
    print(f"{name}: test RMSE={rmse:.3f}  MAE={mae:.3f}")


total_points — best val RMSE: 16.9544
  params:
    colsample_bytree: 0.7
    learning_rate: 0.05
    max_depth: 4
    min_child_weight: 20
    reg_alpha: 0.05
    reg_lambda: 0.2
    subsample: 0.85

winning_margin — best val RMSE: 9.6391
  params:
    colsample_bytree: 1.0
    learning_rate: 0.02
    max_depth: 6
    min_child_weight: 10
    reg_alpha: 0.1
    reg_lambda: 0.0
    subsample: 1.0
total_points: test RMSE=14.411  MAE=10.778
winning_margin: test RMSE=9.340  MAE=7.941
